## 1. Setup Environment
### Tutorial: [Unsloth Deepseek-OCR](https://docs.unsloth.ai/models/deepseek-ocr-how-to-run-and-fine-tune#unsloth-run-deepseek-ocr-tutorial) 

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

In [2]:
# Config
BASE_DIR = "/kaggle/input/nlp-lab02"
DATASET_DIR = os.path.join(BASE_DIR, "dataset/processed")
BASE_MODEL_DIR = os.path.join(BASE_DIR, "models/deepseek-ocr")
FINETUNED_MODEL_DIR = os.path.join(BASE_DIR, "models/deepseek-ocr-finetuned")

OUTPUT_DIR = "/kaggle/working/"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
EVALUATION_OLD_PATH = os.path.join(RESULTS_DIR, "evaluation_deepseek-ocr.csv")
EVALUATION_NEW_PATH = os.path.join(RESULTS_DIR, "evaluation_deepseek-ocr_finetuned.csv")
SUMMARY_PATH = os.path.join(RESULTS_DIR, "summary.csv")

INSTRUCTION = "<image>\nOCR Tiếng Việt."

# Creat folder
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import AutoModel

import os, json, shutil, contextlib, io
import pandas as pd
from jiwer import cer
from PIL import Image

os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit", # Qwen 3 vision support
    "unsloth/Qwen3-VL-8B-Thinking-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Thinking-bnb-4bit",
] # More models at https://huggingface.co/unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-27 07:44:28.656602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766821469.009372      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766821469.116476      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766821469.931982      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766821469.932024      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766821469.932027      24 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


## 2. Load Dataset

In [4]:
from datasets import load_from_disk

dataset = load_from_disk(DATASET_DIR)

test_data = dataset["test"]
print(f"Testing samples: {len(test_data)}")

Testing samples: 201


## 3. Evaluate Model

In [5]:
# Metric: CER (Character Error Rate)
def evaluate(test_data, model, tokenizer):
    results = []

    temp_image_path = "temp_image.jpg"
    temp_output_dir = "temp_inference"
    
    for sample in test_data:
        idx = sample["id"]
        image = sample["image"]
        ground_truth = sample["text"]
        
        image.save(temp_image_path)

        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            model.infer(
                tokenizer, 
                prompt=INSTRUCTION, 
                image_file=temp_image_path,
                output_path=temp_output_dir,
                base_size=1024, 
                image_size=640, 
                crop_mode=True,
                test_compress=False
            )
            
        prediction = str(f.getvalue())
        if "=====================" in prediction:
            prediction = prediction.split("=====================")[-1].strip()
            
        prediction = re.sub(r'\n\s*\n', '\n', prediction)
        prediction = prediction.strip()
        
        error_rate = cer(ground_truth, prediction)

        results.append({
            "id": idx,
            "prediction": prediction,
            "ground_truth": ground_truth,
            "cer": error_rate
        })

    if os.path.exists(temp_image_path):
        os.remove(temp_image_path)

    if os.path.exists(temp_output_dir):
        shutil.rmtree(temp_output_dir)

    return pd.DataFrame(results)

### Evualate with Base Deepsekk-OCR Model

In [6]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = BASE_MODEL_DIR,
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    auto_model = AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)
FastVisionModel.for_inference(model)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at /kaggle/input/nlp-lab02/models/deepseek-ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DeepseekOCRForCausalLM(
  (model): DeepseekOCRModel(
    (embed_tokens): Embedding(129280, 1280)
    (layers): ModuleList(
      (0): DeepseekV2DecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (v_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (o_proj): Linear(in_features=1280, out_features=1280, bias=False)
        )
        (mlp): DeepseekV2MLP(
          (gate_proj): Linear(in_features=1280, out_features=6848, bias=False)
          (up_proj): Linear(in_features=1280, out_features=6848, bias=False)
          (down_proj): Linear(in_features=6848, out_features=1280, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): DeepseekV2RMSNorm((1280,), eps=1e-06)
        (post_attention_layernorm): DeepseekV2RMSNorm((1280,), eps=1e-06)
      )
      (1-11): 11 x DeepseekV2DecoderLayer(

In [7]:
df_old = evaluate(test_data, model, tokenizer)
display(df_old.sort_values(by="cer", ascending=False).head(5))
df_old.to_csv(EVALUATION_OLD_PATH, index=False)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


,id,prediction,ground_truth,cer
187,UIT_HWDB_line/255/11.jpg,“Chúng ta không biết chúng ta không biết chúng...,"công trên "" chiến trường "" bếp núc, nhưng lại ...",1.494737
27,UIT_HWDB_line/251/28.jpg,,tái cấu trúc các ngành công nghiệp.,1.000000
39,UIT_HWDB_line/251/40.jpg,,trình 3 giảm...,1.000000
40,UIT_HWDB_line/254/1.jpg,,KHÁI QUÁT VỀ BIỂN ĐẢO VIỆT NAM,1.000000
35,UIT_HWDB_line/251/36.jpg,,sách cho giáo dục - đào tạo.,1.000000


In [8]:
print(df_old["prediction"][0])

Chúng trình hành động thực hiện Nghị quyết trung ương 9- thỏa.


### Evualate with Base Deepsekk-OCR Finetuned Model

In [9]:
model.load_adapter(FINETUNED_MODEL_DIR)

In [10]:
df_new = evaluate(test_data, model, tokenizer)
display(df_new.sort_values(by="cer", ascending=False).head(5))
df_new.to_csv(EVALUATION_NEW_PATH, index=False)

,id,prediction,ground_truth,cer
183,UIT_HWDB_line/255/7.jpg,tố.,ngờ,1.000000
94,UIT_HWDB_line/253/34.jpg,"những nỗ lực mới để có nhu cầu người làm, làm ...",những sai phạm đó. Có như vậy niềm tin vào lẽ ...,0.913043
73,UIT_HWDB_line/253/13.jpg,hơn thông ơ.,hiệu hiệp sĩ.,0.692308
66,UIT_HWDB_line/253/6.jpg,"nhiều hơn địa phương để tiến bộ nơi chôn bê, c...","nhiều bạn đọc gọi đến bày tỏ sự chia sẻ, ủng hộ.",0.604167
50,UIT_HWDB_line/254/11.jpg,một biện?,vùng biển.,0.600000


### Comparison between Base Model with Finetuned Model

In [11]:
summary = {
    "Model": ["DeepSeek-OCR", "DeepSeek-OCR Fine-tuned"],
    "Samples": [len(df_old), len(df_new)],
    "Average CER": [df_old["cer"].mean(), df_new["cer"].mean()],
    "Median CER": [df_old["cer"].median(), df_new["cer"].median()],
    "Min CER": [df_old["cer"].min(), df_new["cer"].min()],
    "Max CER": [df_old["cer"].max(), df_new["cer"].max()]
}

df_summary = pd.DataFrame(summary)
df_summary.to_csv(SUMMARY_PATH, index=False)
display(df_summary)

,Model,Samples,Average CER,Median CER,Min CER,Max CER
0,DeepSeek-OCR,201,0.571008,0.406780,0.0,1.494737
1,DeepSeek-OCR Fine-tuned,201,0.139329,0.067416,0.0,1.000000
